# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the dataset's Croissant schema. All references use `@id` values.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record Set '@id': {record_set['@id']}")
    # List fields inside record set
    if 'field' in record_set:
        fields = record_set['field']
        print("  Fields:")
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            # Fields may be @id reference or dict; ensure we print @id
            if isinstance(field, dict) and '@id' in field:
                print(f"    - {field['@id']}")
            elif isinstance(field, str):
                print(f"    - {field}")
        print()
    else:
        print("  No fields found.\n")

## 3. Data Extraction
Load records from each available record set into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview for precise access.

In [ ]:
# Create DataFrames for each record set
# Collect all record_set @id's
record_set_ids = [r['@id'] for r in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from record set {rs_id}")
        else:
            print(f"No records found for record set {rs_id}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# Preview the columns of the main record set (choose the primary one by size or by inspecting available)
if dataframes:
    primary_rs = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"\nPrimary record set '@id': {primary_rs}")
    print("Columns:", dataframes[primary_rs].columns.tolist())
    display(dataframes[primary_rs].head())
else:
    print("No dataframes loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping using `@id` references for fields.

In [ ]:
# --- Example EDA on a numeric field ---
# For this dataset, likely numeric fields might include `schema:age`, `schema:interval`, or similar. Let's print numeric-like columns:
import numpy as np

if dataframes:
    df = dataframes[primary_rs]
    print("Available columns (with types):")
    print(df.dtypes)
    # Try to infer a likely numeric field from the DataFrame
    numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if numeric_candidates:
        print("\nCandidate numeric fields:", numeric_candidates)
        numeric_field = numeric_candidates[0]
    else:
        # Try to parse for numeric columns (e.g., 'age', 'interval')
        possible_numeric_keywords = ['age', 'interval', 'count', 'duration', 'year']
        numeric_field = None
        for col in df.columns:
            if any(kw in col.lower() for kw in possible_numeric_keywords):
                try:
                    # Try to convert column to numeric
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].notnull().any():
                        numeric_field = col
                        break
                except:
                    pass
        if numeric_field is None:
            print("No obvious numeric field found for EDA.")
    
    if numeric_field:
        print(f"\nUsing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a likely categorical field (try 'sex', 'gender', 'location', etc)
        group_candidates = [col for col in df.columns if any(kw in col.lower() for kw in ['sex', 'gender', 'location', 'anatomical', 'comorbidity'])]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping by field: {group_field}\n")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df)
        else:
            group_field = None
            print("No suitable group field found.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df, palette="Set2")
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary:**

- Successfully loaded clinical and molecular records relating to second primary colorectal cancer survivors.
- Explored the record set(s) and field structure using their unique `@id` values for unambiguous referencing.
- Performed initial exploratory data analysis, filtered/normalized a numeric variable, and grouped by a key attribute if available.
- Visualizations reveal main distribution trends and cohort characteristics. For deeper insights, consult the Croissant schema documentation for clinical variable definitions and leverage further domain-specific analysis methods.

**Note:** This notebook serves as a starting point for working with FAIR² datasets and can be extended for downstream biomedical, statistical, or ML modeling tasks as required.